<a href="https://colab.research.google.com/github/moodleyrivan-tech/savings-goal-coach-ISYS2007/blob/main/finance_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Savings Goal Coach

Problem: Many pepole save a savings goal. Especially my demographic being young pepole who save for things like trips or emergency funds. However most pepole do not have easy way to check whether their current income and spending will actually get them to there target. I made this savings goal coach to prevent that and support positive spending habits

User (Demographic):A young student or individual in there 20's with a specific savings goal and a rough idea of their monthly income and expenses, who wants a answer on whether they're on track to a small goal or even a big goal like saving up for a house.

What this app does: The individual frst enters their goal, then their savings, timeframe in which they want to achive their goal, their income
and expenses. The app will then calculate their required monthly saving rate, and compares it to their actual income. Then the app lets them ask an finance coach (Gemini) which can help answer any svings answers they need.

**six-step design**

1. Understand the problem
A young person, specifically my age democgraphic individuals in their 20's who usally have long term savings goals (a trip, an emergency fund, a house deposit) and wants to know if they are on track to reach it by their target date. If they are not, they want to know what to change. The app should explain this in plain, encouraging language, not just output a number. I want this app to support good spending habits and aid the user in achieving their goal with a little left extra to spare.


2. Inputs and outputs
- Inputs: goal amount, current savings, months remaining, monthly income, monthly expenses
- Outputs: required monthly saving, actual monthly surplus, on-track status, the gap between the two, and a coach reply that uses those numbers


3. Worked example by hand. This is specifically for my case as I am going to japan so I am going to workout out how the math should look like so I know what the code should print. This is not representative of the users actual goal rather its just to undertand how the code will work:

Goal: \$5,000. Current savings: \$800. Months remaining: 10. Income: \$4,200. Expenses: \$3,600.
- Required monthly saving = (5000 - 800) / 10 = $420
- Actual monthly surplus = 4200 - 3600 = $600
- 600 is more than 420, so I am on track to the goal with $180 a month to spare.


4. Pseudocode (Rough plan on how the code will work)

function calculate_savings_plan(goal, current_savings, months_remaining, income, expenses):

    if months_remaining <= 0: raise error
    if goal < 0 or current_savings < 0: raise error
    if current_savings >= goal: return "goal already met"
    required_monthly = (goal - current_savings) / months_remaining
    surplus = income - expenses
    on_track = surplus >= required_monthly
    return required_monthly, surplus, on_track, surplus - required_monthly



5. Convert to Python
The pseudocode becomes the calculate_savings_plan function below. The Gemini coach and the Gradio interface are built on top of it in the later sections. This allows the Ai coach to use User information and support those positive spending habits. (convert plan to actual code)


6. Test with a variety of data
After i have done all the code I will then test the code to see if it works in various diffrent circumstances. The tests section will check normal cases (on track, behind target), cases like ones were the goal is already met and bad cases and input (zero months, negative amounts).

In [33]:
!pip install -q google-generativeai gradio

**Pseudocode**

The pseudocoden written from step 4 will now be written as Python. Meaning that it must work out whether the user is on track and handles bad input (negative numbers, zero time left, a goal that is already met) instead of crashing.

In [34]:
def calculate_savings_plan(goal, current_savings, months_remaining,
                            monthly_income, monthly_expenses):
    """Work out whether a user is on track to hit a savings goal.

    Returns a dictionary with the required monthly saving, the user's actual
    monthly surplus, whether they're on track, and the size of the gap.
    Raises ValueError on invalid input (zero/negative months, negative amounts).
    """
    if months_remaining <= 0:
        raise ValueError("months_remaining must be greater than 0")
    if goal < 0 or current_savings < 0:
        raise ValueError("goal and current_savings cannot be negative")

    if current_savings >= goal:
        return {
            "required_monthly": 0,
            "actual_monthly_surplus": monthly_income - monthly_expenses,
            "on_track": True,
            "gap": None,
            "message": "Goal already met"
        }

    remaining_needed = goal - current_savings
    required_monthly = remaining_needed / months_remaining
    actual_monthly_surplus = monthly_income - monthly_expenses
    on_track = actual_monthly_surplus >= required_monthly
    gap = actual_monthly_surplus - required_monthly

    return {
        "required_monthly": round(required_monthly, 2),
        "actual_monthly_surplus": round(actual_monthly_surplus, 2),
        "on_track": on_track,
        "gap": round(gap, 2),
        "message": "On track" if on_track else "Behind target"
    }

**Calculator: Ai coach**

After creating the base for the finance calculator I will know create the AI coach (made with Gemini). This means I need to get a free API key from Google AI Studio and add it to Colab Secrets (GEMINI_API_KEY). The name of the coach is sage and it is an finance coach. Its system instruction sets its persona and purpose to target my demographic. This means its a casual, encouraging coach for students and young people in their 20s, who have a specific savings goal in mind.


In [35]:
from google.colab import userdata
import google.generativeai as genai

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))

SYSTEM_INSTRUCTION = """
Your name is Sage, a friendly, realistic personal finance coach inside a savings-goal app.
Your users (main demographic) are students and young people in their 20s saving for goals like trips,
emergency funds, or a house deposit (Just to name a few). You must Speak casually and encouragingly, and support
positive spending habits without judging them.
Use the user's actual numbers, not generic advice. If they're behind target, say so
honestly and suggest one concrete adjustment, such as a smaller cut to a specific
expense or extending their timeframe. If asked something unrelated to personal finance,
redirect them back gently. Keep replies under 4 sentences unless asked for more detail.
Additionally you may use emojis where appropriate and use casual everyday langauge in your responses.
"""

model = genai.GenerativeModel(
    model_name="gemini-3.5-flash-lite",
    system_instruction=SYSTEM_INSTRUCTION
)

In [36]:
from google.api_core.exceptions import ResourceExhausted, ServiceUnavailable, DeadlineExceeded

def ask_coach(user_message, plan):
    """Send the user's question to Gemini along with their real savings numbers,
    so the reply is based on their actual data rather than generic advice."""
    context = f"""
    User's savings goal details:
    - Required monthly saving: ${plan['required_monthly']}
    - Actual monthly surplus: ${plan['actual_monthly_surplus']}
    - Status: {plan['message']}
    - Gap: {plan['gap'] if plan['gap'] is not None else 'N/A'}

    User's question: {user_message}
    """
    try:
        response = model.generate_content(context, request_options={"timeout": 300})
        return response.text
    except ResourceExhausted:
        return "Sage is getting a lot of questions right now. Please wait a minute and try again."
    except ServiceUnavailable:
        return "Sage's servers are really busy right now. Please try again in a minute."
    except DeadlineExceeded:
        return "Sage took too long to answer. Please try again in a minute."
    except Exception as e:
        return f"Sage couldn't answer right now ({type(e).__name__}). Please try again."

In [37]:
test_plan = calculate_savings_plan(5000, 800, 10, 4200, 3600)
try:
    print(ask_coach("Should I cut back on coffee to hit my goal faster?", test_plan))
except Exception as e:
    print(f"Could not reach the coach at the moment due to an error: {e}\nPlease try running this cell again.")

Hey! Since you've actually got a $600 monthly surplus against your $420 goal, you're already totally on track! 🚀 Honestly, you don't *need* to cut back on coffee unless you genuinely want to speed things up even more. Keep enjoying your daily brew guilt-free, because your current budget is already totally crushing it! ☕✨


**Calculator: expanding on interface**

The Gradio interface ties the calculator and the coach together so a user can use the app: enter your numbers, see the plan, and optionally ask Sage a question (The AI coach). If a field is left blank, the app shows an error message instead of crashing.

In [38]:
import gradio as gr

def run_assistant(goal, current_savings, months_remaining,
                   monthly_income, monthly_expenses, user_message):
    fields = [goal, current_savings, months_remaining, monthly_income, monthly_expenses]
    if any(f is None for f in fields):
        return "Error: please fill in every number field before submitting."

    try:
        plan = calculate_savings_plan(goal, current_savings, months_remaining,
                                       monthly_income, monthly_expenses)
    except ValueError as e:
        return f"Error: {e}"

    summary = (
        f"Required monthly saving: ${plan['required_monthly']}\n"
        f"Your actual monthly surplus: ${plan['actual_monthly_surplus']}\n"
        f"Status: {plan['message']}\n"
    )

    if user_message and user_message.strip():
        return summary + "\nSage says: " + ask_coach(user_message, plan)
    return summary

demo = gr.Interface(
    fn=run_assistant,
    inputs=[
        gr.Number(label="Savings Goal ($)"),
        gr.Number(label="Current Savings ($)"),
        gr.Number(label="Months Remaining"),
        gr.Number(label="Monthly Income ($)"),
        gr.Number(label="Monthly Expenses ($)"),
        gr.Textbox(label="Ask Sage a question (optional)"),
    ],
    outputs="text",
    title="Savings Goal Coach",
    description="Enter your goal details to see if you're on track, and ask Sage for advice."
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e44b1bd5cac39010b3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Testing the calculator**

Now that the code is written I will test with a variety of data to confirm wether the project works. This mean it will test various iputs as stated above like: normal cases (on track, behind target), cases like ones were the goal is already met and bad cases and input (zero months, negative amounts).

In [39]:
# Normal case: on track
r = calculate_savings_plan(5000, 800, 10, 4200, 3600)
assert r["required_monthly"] == 420.0
assert r["actual_monthly_surplus"] == 600.0
assert r["on_track"] == True
assert r["gap"] == 180.0

# Normal case: behind target
r2 = calculate_savings_plan(10000, 0, 5, 3000, 2900)
assert r2["required_monthly"] == 2000.0
assert r2["on_track"] == False

# Edge case: expenses higher than income (negative surplus)
r3 = calculate_savings_plan(1200, 0, 6, 2000, 2500)
assert r3["actual_monthly_surplus"] == -500.0
assert r3["on_track"] == False
assert r3["gap"] == -700.0

# Edge case: surplus exactly equals what is needed
r4 = calculate_savings_plan(1200, 0, 12, 2000, 1900)
assert r4["on_track"] == True
assert r4["gap"] == 0.0

# Edge case: decimal amounts
r5 = calculate_savings_plan(999.99, 0, 3, 2000, 1600)
assert r5["required_monthly"] == 333.33
assert r5["gap"] == 66.67

# Edge case: goal already met (more saved than needed, and exactly equal)
r6 = calculate_savings_plan(1000, 1500, 6, 3000, 2800)
assert r6["message"] == "Goal already met"
r7 = calculate_savings_plan(500, 500, 4, 3000, 2800)
assert r7["message"] == "Goal already met"

# Bad input: zero months should raise an error
try:
    calculate_savings_plan(5000, 0, 0, 3000, 2000)
    assert False, "Should have raised ValueError"
except ValueError:
    pass

# Bad input: negative goal should raise an error
try:
    calculate_savings_plan(-100, 0, 5, 3000, 2000)
    assert False, "Should have raised ValueError"
except ValueError:
    pass

# Bad input: negative current savings should raise an error
try:
    calculate_savings_plan(5000, -50, 10, 3000, 2000)
    assert False, "Should have raised ValueError"
except ValueError:
    pass

print("All calculator tests passed.")

All calculator tests passed.


In [40]:
# App test: a blank field gives a friendly error instead of crashing
assert run_assistant(None, 800, 10, 4200, 3600, "") == "Error: please fill in every number field before submitting."

# App test: zero months gives a clear error message
assert "months_remaining must be greater than 0" in run_assistant(5000, 800, 0, 4200, 3600, "")

# App test: a negative goal gives a clear error message
assert "cannot be negative" in run_assistant(-100, 0, 5, 3000, 2000, "")

# App test: a normal case shows the right numbers and status
output = run_assistant(5000, 800, 10, 4200, 3600, "")
assert "Required monthly saving: $420.0" in output
assert "On track" in output

print("All app tests passed.")

All app tests passed.
